# Equilibrium Fairness — Hiring Demo

**Dataset.** UCI Adult (synthetic stand-in if no network).
**Metric.** Demographic parity gap on sex.
**Threshold.** θ = 0.05 (5 percentage points).

This notebook walks through the four stages of the Equilibrium Fairness loop with one window's data made visible at each step.

## Stage 1 — Initialise (E₀)

In [ ]:
import yaml
from pathlib import Path
from src.loader import load_dataset

cfg = yaml.safe_load(Path('../configs/hiring_adult.yaml').read_text())
data = load_dataset(cfg)

print('Features retained (E₀-equalised):')
for f in data.feature_names[:10]:
    print(f'  - {f}')
print(f'  ... ({len(data.feature_names)} features total)')
print()
print('Features dropped by E₀ policy:', data.e0_dropped)
print()
print(f'Train size: {len(data.X_train)},  Test size: {len(data.X_test)}')

Note that `sex` and `race` are **removed from the feature matrix** but `data.group_test` retains the `sex` label for monitoring purposes. This is the architectural separation: the model never sees the protected attribute; the monitor always does.

## Train the baseline

A standard logistic-regression classifier on the equalised inputs. The model's job is to make decisions; the architecture's job is to monitor the decisions for drift.

In [ ]:
from src.baseline import fit_baseline, predict_rolling

model = fit_baseline(data.X_train, data.y_train, seed=cfg['random_seed'])
print('Baseline accuracy on test set:', model.score(data.X_test, data.y_test))

## Stages 2–4 — one window at a time

We deliberately run the loop window-by-window here, with commentary, so the four stages of the architecture are visible.

In [ ]:
from src.monitor import compute_drift
from src.threshold import check
from src.escalate import dispatch

log_path = Path('../logs/hiring_notebook.jsonl')
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text('')  # fresh log

for i, window in enumerate(predict_rolling(
    model, data.X_test, data.y_test, data.group_test,
    window_size=cfg['window_size'],
)):
    # Stage 2: compute drift
    reading = compute_drift(window, cfg)
    # Stage 3: threshold check
    result = check(reading.magnitude, cfg['theta'])
    # Stage 4: escalation event
    event = dispatch(reading, result, cfg, log_path)

    print(f'Window {i}: rows {reading.window_start}-{reading.window_end}')
    print(f'  |δ| = {abs(reading.magnitude):.4f}  vs θ = {cfg["theta"]}')
    print(f'  breach: {result.breach}')
    if result.breach:
        print(f'  → escalated to κ-owner: {event["kappa_owner"]}')
    print()

## What the log looks like

Each window writes one structured event. Breaches are flagged. **No automatic correction has been performed** — the architecture's load-bearing principle is *monitor ≠ decide*. Correction is the responsibility of the named κ-owner.

In [ ]:
import json
for line in log_path.read_text().strip().split('\n'):
    event = json.loads(line)
    print(json.dumps(event, indent=2))
    print()

## Interpretation

The architecture has done its job: detected drift, applied a pre-agreed threshold, written an auditable event with a named human owner. The next step — the actual *correction* — happens outside this loop. That separation is the point.